# Imports and settings

In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil

# --- Color Constants ---
GREEN = '\033[92m'
YELLOW = '\033[93m'
RED = '\033[91m'
RESET_COLORS = '\033[0m'

# --- Paths Configuration ---
# Notebook is in root, so we enter 'data' directly
DATA_DIR = 'data'
PLOTS_DIR = 'plots'

RAW_DATA_NPY = os.path.join(DATA_DIR, 'Dropbox_Dataset.npy')
CLEAN_DATA_NPY = os.path.join(DATA_DIR, 'dataset_clean.npy')

# --- Plot File Paths ---
USER_DIST_PLOT = os.path.join(PLOTS_DIR, 'user_rating_dist.png')
FILTERED_USER_DIST_PLOT = os.path.join(PLOTS_DIR, 'filtered_user_dist.png')
COMPARISON_PLOT = os.path.join(PLOTS_DIR, 'original_vs_filtered.png')
MOVIE_DIST_PLOT = os.path.join(PLOTS_DIR, 'movie_rating_dist.png')

# --- Analysis Constants ---
TOP_N_THRESHOLD = 10000
MIN_RATINGS_N = 10

In [43]:
%matplotlib tk

## Initializations and Directory Setup

In [44]:
# --- Safety Check & Directory Setup ---
if not os.path.exists(PLOTS_DIR):
    os.makedirs(PLOTS_DIR)
    print(f"{GREEN}Directory 'plots' is ready.{RESET_COLORS}")

try:
    dataset = np.load(RAW_DATA_NPY, allow_pickle=True)
    print(f"{GREEN}Dataset loaded successfully from {RAW_DATA_NPY}{RESET_COLORS}")

    # --- Unpacking the Wrapped Data ---
    # Since print(df.columns) showed only 1 column, we extract the inner list from each row
    # This converts (4669820, 1) into (4669820, 4)
    unpacked_data = np.array(dataset.tolist())

    # Now create the DataFrame with the correct columns
    df = pd.DataFrame(unpacked_data, columns=['user', 'movie', 'rating', 'date'])

    print(f'\n{YELLOW}DataFrame info:{RESET_COLORS}')
    print(f'- Total ratings: {len(df)}')
    print(f'- Unique users: {df["user"].nunique()}')
    print(f'- Unique movies: {df["movie"].nunique()}')
    print('-' * 30)

except Exception as e:
    print(f"{RED}Error during loading/unpacking: {e}{RESET_COLORS}")

Dataset loaded successfully from data/Dropbox_Dataset.npy
Error during loading/unpacking: Shape of passed values is (4669820, 1), indices imply (4669820, 4)


# User Rating Distribution Analysis

## Binning and Statistics

In [ ]:
# Get the number of ratings per user
ratings_per_user = df.groupby('user').size()

# Slice the possible rating counts in logical bins
bins = [0, 1, 2, 3, 10, 20, 100, float('inf')]
labels = ['1', '2', '3', '4-10', '11-20', '21-100', '100+']

# Put each user in the respective bin
user_bins = pd.cut(ratings_per_user, bins=bins, labels=labels, right=True)
bin_counts = user_bins.value_counts().sort_index()

print(f'\n{YELLOW}Percentage of users in each rating count bin:{RESET_COLORS}')
print((bin_counts / len(ratings_per_user) * 100).round(2))

In [ ]:
### Visualizing User Distribution

In [ ]:
plt.figure(figsize=(10, 6))
(bin_counts / 100000).plot(kind='bar')
plt.title('Number of Users by Rating Count')
plt.xlabel('Number of Ratings per User')
plt.ylabel('Number of Users (×100,000)')
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(USER_DIST_PLOT)
plt.show()

# Dimensionality Reduction: Lonely Ratings Analysis

## Identifying Single Ratings

In [ ]:
# Count movie ratings
movie_rating_counts = df['movie'].value_counts()

# Find movies and users with exactly one rating
movies_rated_once = movie_rating_counts[movie_rating_counts == 1].index.to_list()
one_rating_users = ratings_per_user[ratings_per_user == 1].index.to_list()

# Identify "lonely ratings" (intersection)
lonely_ratings_mask = (df['user'].isin(one_rating_users)) & (df['movie'].isin(movies_rated_once))
lonely_ratings = df[lonely_ratings_mask]

print(f'\n{RED}Movies rated exactly once: {len(movies_rated_once)}{RESET_COLORS}')
print(f'{RED}Users with only one rating: {len(one_rating_users)}{RESET_COLORS}')
print(f'{RED}Lonely ratings count: {len(lonely_ratings)}{RESET_COLORS}')

# Filtered Dataset Comparison

## Data Filtering and Re-binning

In [ ]:
# Remove movies rated once
df_movies_multiple_ratings = df[~df['movie'].isin(movies_rated_once)]

# Repeat distribution for filtered dataset
ratings_per_user_filtered = df_movies_multiple_ratings.groupby('user').size()
user_bins_filtered = pd.cut(ratings_per_user_filtered, bins=bins, labels=labels, right=True)
bin_counts_filtered = user_bins_filtered.value_counts().sort_index()

# Comparison Table
comparison_df = pd.DataFrame({'Original': bin_counts, 'Filtered': bin_counts_filtered}).fillna(0)
print(f'\n{YELLOW}Comparison of user counts:{RESET_COLORS}')
print(comparison_df)

### Plotting Original vs Filtered Data

In [ ]:
plt.figure(figsize=(12, 7))
x = np.arange(len(bin_counts.index))
width = 0.35

original_bars = plt.bar(x - width/2, bin_counts.values/100000, width, label='Original', color='skyblue', alpha=0.7)
filtered_bars = plt.bar(x + width/2, bin_counts_filtered.reindex(bin_counts.index, fill_value=0)/100000, width, label='Filtered', color='lightcoral', alpha=0.7)

plt.title('Comparison: Users by Rating Count (Original vs Filtered)', fontweight='bold')
plt.xticks(x, bin_counts.index, rotation=45)
plt.legend()
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()

plt.savefig(COMPARISON_PLOT)
plt.show()

# Final Movie Distribution and Thresholding

## Movie Rating Distribution

In [ ]:
movie_bins = [0, 1, 2, 3, 5, 10, 20, 50, 100, float('inf')]
movie_labels = ['1', '2', '3', '4-5', '6-10', '11-20', '21-50', '51-100', '100+']

movie_rating_bins = pd.cut(movie_rating_counts, bins=movie_bins, labels=movie_labels, right=True)
movie_bin_counts = movie_rating_bins.value_counts().sort_index()

plt.figure(figsize=(10,6))
(movie_bin_counts / 1000).plot(kind='bar', color='mediumseagreen')
plt.title('Movie Distribution by Number of Ratings')
plt.xlabel('Number of Ratings per Movie')
plt.ylabel('Number of Movies (×1,000)')
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(MOVIE_DIST_PLOT)
plt.show()

## Top N Movie Filtering (R_min = 1)

In [ ]:
df_users_multiple_ratings = df[~df['user'].isin(one_rating_users)]
movie_ratings_count_after_r1 = df_users_multiple_ratings['movie'].value_counts()
top_n_movies = movie_ratings_count_after_r1.nlargest(TOP_N_THRESHOLD).index

df_top_movies = df_users_multiple_ratings[df_users_multiple_ratings['movie'].isin(top_n_movies)]

print(f'\n{GREEN}Top {TOP_N_THRESHOLD} Movies filtering results:{RESET_COLORS}')
print(f'Remaining Ratings: {len(df_top_movies)}')
print(f'Minimum ratings in remaining movies: {df_top_movies.groupby("movie").size().min()}')

## Advanced Filtering (Users with ≥ N Ratings)

In [ ]:
# Filter users by N ratings first
at_least_n_rating_users = ratings_per_user[ratings_per_user >= MIN_RATINGS_N].index.to_list()
df_users_filtered = df[df['user'].isin(at_least_n_rating_users)]

# Then take top movies
movie_ratings_count_after_r1_n = df_users_filtered['movie'].value_counts()
top_n_movies_second = movie_ratings_count_after_r1_n.nlargest(TOP_N_THRESHOLD).index
df_top_movies_second = df_users_filtered[df_users_filtered['movie'].isin(top_n_movies_second)]

print(f'\n{GREEN}Strict filtering (Users >= {MIN_RATINGS_N}) results:{RESET_COLORS}')
print(f'Unique users remaining: {df_top_movies_second["user"].nunique()}')
print(f'Total ratings: {len(df_top_movies_second)}')